# PHÂN TÍCH XU HƯỚNG TUYỂN DỤNG IT TẠI VIỆT NAM

## 1. Phát biểu bài toán
Bài toán của nhóm là **phân tích dữ liệu nhằm khảo sát xu hướng tuyển dụng IT tại Việt Nam** từ dữ liệu do nhóm tự crawl, qua đó đánh giá khả năng sử dụng các biến/đặc trưng `X_i` để khám phá cấu trúc thị trường tuyển dụng.

Trong giai đoạn hiện tại, nhóm **không chọn một biến mục tiêu `Y` cụ thể để dự đoán** làm trọng tâm chính, vì dữ liệu mức lương bị thiếu khá nhiều và chưa đủ ổn định để xây dựng một bài toán hồi quy đáng tin cậy. Do đó, notebook được định hướng theo:
- phân tích mô tả (descriptive analytics) về thị trường tuyển dụng, và
- khảo sát cấu trúc dữ liệu theo hướng khám phá xu hướng; nếu cần mở rộng về sau thì đây là nền tốt cho **phân cụm (clustering)**.

Tuy nhiên, **thống kê mức lương vẫn được giữ lại như một nhánh phân tích phụ** để minh họa thêm cho bức tranh thị trường lao động IT, ví dụ so sánh lương theo level, địa điểm hoặc nhóm nghề trên tập con các mẫu có công khai lương.

Notebook này tập trung trả lời các câu hỏi chính:
- Nhóm vị trí nào được tuyển nhiều nhất?
- Kỹ năng nào xuất hiện nhiều nhất?
- Những skill nào thường đi cùng nhau?
- Thành phố nào tuyển nhiều cho từng nhóm nghề?
- Seniority level nào gắn với từng nhóm skill?
- Trên tập con có lương, mức lương phân bố như thế nào theo level hoặc nhóm nghề?

**Lưu ý:** trọng tâm của bài giữa kỳ là dữ liệu, cleaning, encoding/NLP, feature engineering, trực quan hóa và kết luận. Phần mô hình hóa nếu có chỉ là định hướng cho bài tập theo sau.

### Định hướng bài toán trên bộ dữ liệu hiện tại
- Chưa chốt biến mục tiêu `Y` cho supervised learning làm trọng tâm chính.
- Hướng phân tích phù hợp nhất hiện tại: **phân tích xu hướng tuyển dụng**.
- Mức lương được giữ lại như **biến tham khảo bổ sung**, không phải trục chính của notebook.
- Nếu cần diễn giải theo khuôn khổ học máy: dữ liệu hiện tại phù hợp hơn với **khám phá dữ liệu** và có thể mở rộng sang **clustering** ở bước sau.
- Nguồn dữ liệu: nhóm **tự crawl** từ website tuyển dụng, không tải dataset có sẵn.
- Quy mô raw data hiện tại: `200` tin từ ITViec và `809` tin từ TopCV, tổng cộng `1009` mẫu raw, thỏa yêu cầu `>1000` mẫu.
- Sau cleaning và loại trùng lặp còn `780` mẫu sạch; trong đó một phần mẫu có `salary_avg` để dùng cho thống kê minh họa về lương.

## 2. Thiết lập dữ liệu và môi trường
Phần này nạp các tập dữ liệu hiện có trong project, đồng thời import các hàm xử lý dữ liệu và NLP để notebook bám sát đúng pipeline hiện tại của nhóm.


In [ ]:
from pathlib import Path
import warnings
from itertools import combinations
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.manifold import TSNE
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src.processing.clean_jobs import (
    normalize_location,
    normalize_remote_option,
    parse_experience_years,
)
from src.processing.extract_skills import enrich_with_skill_features

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11

BASE_DIR = Path('.')
RAW_ITVIEC = BASE_DIR / 'data/raw/itviec_jobs_20260315_115039.csv'
RAW_TOPCV = BASE_DIR / 'data/raw/topcv_jobs_20260314_231436.csv'
CLEAN_PATH = BASE_DIR / 'data/clean-data/jobs_cleaned.csv'
FEATURE_PATH = BASE_DIR / 'data/clean-data/jobs_features.csv'

df_itviec = pd.read_csv(RAW_ITVIEC)
df_topcv = pd.read_csv(RAW_TOPCV)
df_raw_merged = pd.concat(
    [df_itviec.assign(source='itviec'), df_topcv.assign(source='topcv')],
    ignore_index=True,
)
df_clean = pd.read_csv(CLEAN_PATH)
df_features = pd.read_csv(FEATURE_PATH)

summary_loaded = pd.DataFrame(
    {
        'dataset': ['ITViec raw', 'TopCV raw', 'jobs_cleaned', 'jobs_features'],
        'rows': [len(df_itviec), len(df_topcv), len(df_clean), len(df_features)],
        'columns': [df_itviec.shape[1], df_topcv.shape[1], df_clean.shape[1], df_features.shape[1]],
        'path': [str(RAW_ITVIEC), str(RAW_TOPCV), str(CLEAN_PATH), str(FEATURE_PATH)],
    }
)
summary_loaded


## 3. Thu thập dữ liệu
Theo yêu cầu học phần, nhóm phải **tự thu thập dữ liệu**. Trong project hiện tại, dữ liệu được crawl từ hai nguồn thực tế đang có trong thư mục `data/raw/`:
- `ITViec`: nền tảng chuyên về tuyển dụng IT, dữ liệu thiên về các vị trí kỹ thuật.
- `TopCV`: số lượng tin lớn hơn, phong phú về vị trí, địa điểm và keyword kỹ năng.

Từ mã nguồn hiện có:
- `src/data_collection/itviec_crawler.py` sử dụng **Selenium + BeautifulSoup** để lấy URL và đi vào trang chi tiết từng tin.
- `src/data_collection/topcv_crawler.py` sử dụng **requests + BeautifulSoup** để crawl trực tiếp từ trang listing.

Điểm quan trọng là dữ liệu này do nhóm tự thu thập, nên đáp ứng đúng yêu cầu “không dùng dataset tải sẵn”.


In [ ]:
collection_summary = pd.DataFrame(
    {
        'Nguồn': ['ITViec', 'TopCV'],
        'File raw': [RAW_ITVIEC.name, RAW_TOPCV.name],
        'Script crawl': [
            'src/data_collection/itviec_crawler.py',
            'src/data_collection/topcv_crawler.py',
        ],
        'Cách thu thập': [
            'Selenium + BeautifulSoup, crawl URL rồi vào trang chi tiết',
            'requests + BeautifulSoup, phân tích HTML ở trang listing',
        ],
        'Số mẫu raw': [len(df_itviec), len(df_topcv)],
        'Số biến raw': [df_itviec.shape[1], df_topcv.shape[1]],
        'Ngày crawl từ tên file': ['2026-03-15 11:50:39', '2026-03-14 23:14:36'],
    }
)
collection_summary.loc[len(collection_summary)] = [
    'Tổng cộng', '-', '-', 'Dữ liệu do nhóm tự crawl từ 2 nguồn', len(df_raw_merged), df_raw_merged.shape[1], '-'
]
collection_summary


### Schema dữ liệu raw
Hai nguồn raw đang có schema khá đồng nhất, thuận lợi cho việc hợp nhất và phân tích:
- `job_title`
- `tech_stack`
- `experience_years`
- `location`
- `company_name`
- `remote_option`
- `salary_min`
- `salary_max`
- `currency`

Dù notebook này không còn đặt trọng tâm vào lương, các trường còn lại vẫn đủ mạnh để phân tích xu hướng nghề nghiệp, địa điểm, level và skill.


In [ ]:
print('Các cột của ITViec:', list(df_itviec.columns))
print('Các cột của TopCV :', list(df_topcv.columns))

print('
Mẫu dữ liệu thô ITViec:')
display(df_itviec.head(3))

print('Mẫu dữ liệu thô TopCV:')
display(df_topcv.head(3))


## 4. Mô tả dữ liệu ban đầu
Trước khi làm sạch, cần nhìn dữ liệu theo ba khía cạnh:
- quy mô mẫu và độ phủ của các biến;
- mức độ thiếu dữ liệu ở các trường quan trọng;
- mức độ không đồng nhất giữa các nguồn, đặc biệt ở `location`, `remote_option`, `experience_years` và `tech_stack`.

Đây là bước cần thiết để xác định xem dữ liệu có đủ tốt cho phân tích xu hướng hay không.


In [ ]:
raw_overview = pd.DataFrame(
    {
        'Chỉ số': [
            'Số mẫu raw sau khi gộp',
            'Số nguồn dữ liệu',
            'Số biến raw',
            'Số mẫu clean',
            'Số mẫu feature-ready',
            'Số mẫu có salary_avg',
            'Tỷ lệ mẫu có salary_avg',
        ],
        'Giá trị': [
            len(df_raw_merged),
            df_raw_merged['source'].nunique(),
            df_raw_merged.shape[1],
            len(df_clean),
            len(df_features),
            int(df_clean['salary_avg'].notna().sum()),
            round(df_clean['salary_avg'].notna().mean(), 4),
        ],
    }
)

missing_ratio_raw = df_raw_merged.isna().mean().sort_values(ascending=False).rename('Tỷ lệ thiếu raw').to_frame()
missing_ratio_clean = df_clean.isna().mean().sort_values(ascending=False).rename('Tỷ lệ thiếu clean').to_frame()

raw_overview

In [ ]:
display(missing_ratio_raw.head(10))
display(missing_ratio_clean.head(10))


## 5. Thống kê mô tả và trực quan hóa đơn biến
Mục tiêu của phần này là mô tả nhanh thị trường tuyển dụng IT thông qua các biến quan trọng nhất cho phân tích xu hướng: `job_title_group`, `level`, `location`, `company_type`, `remote_option`, `source` và `skills_extracted`.

Bên cạnh đó, notebook vẫn giữ **thống kê mức lương ở mức mô tả** trên tập con các mẫu có công khai lương, nhằm bổ sung góc nhìn về thị trường lao động mà không biến lương thành trục chính của bài toán.

In [ ]:
source_counts = df_clean['source'].value_counts()
top_locations = df_clean['location'].fillna('Thiếu').value_counts().head(10)
top_levels = df_clean['level'].fillna('Thiếu').value_counts()
top_roles = df_clean['job_title_group'].fillna('Thiếu').value_counts().head(12)
top_company_types = df_clean['company_type'].fillna('Thiếu').value_counts()
salary_summary = df_clean[['salary_min', 'salary_max', 'salary_avg']].describe().T

display(source_counts.to_frame('Số tin'))
display(top_locations.to_frame('Số tin'))
display(top_levels.to_frame('Số tin'))
display(top_roles.to_frame('Số tin'))
display(top_company_types.to_frame('Số tin'))
display(salary_summary)

In [ ]:
skill_freq = (
    df_features['skills_extracted']
    .fillna('')
    .str.split(', ')
    .explode()
)
skill_freq = skill_freq[skill_freq.ne('')].value_counts().head(15)

df_salary = df_clean.dropna(subset=['salary_avg']).copy()

fig, axes = plt.subplots(2, 3, figsize=(20, 11))

sns.countplot(data=df_clean, x='source', order=source_counts.index, ax=axes[0, 0], palette='Set2')
axes[0, 0].set_title('Số lượng tin theo nguồn')
axes[0, 0].set_xlabel('source')
axes[0, 0].set_ylabel('Số tin')

sns.countplot(
    data=df_clean,
    y='job_title_group',
    order=top_roles.index,
    ax=axes[0, 1],
    color='#4c78a8'
)
axes[0, 1].set_title('Nhóm vị trí được tuyển nhiều nhất')
axes[0, 1].set_xlabel('Số tin')
axes[0, 1].set_ylabel('job_title_group')

sns.countplot(
    data=df_clean,
    y='location',
    order=top_locations.index,
    ax=axes[0, 2],
    color='#f58518'
)
axes[0, 2].set_title('Top địa điểm tuyển dụng nhiều nhất')
axes[0, 2].set_xlabel('Số tin')
axes[0, 2].set_ylabel('location')

sns.barplot(x=skill_freq.values, y=skill_freq.index, ax=axes[1, 0], color='#54a24b')
axes[1, 0].set_title('Top 15 kỹ năng xuất hiện nhiều nhất')
axes[1, 0].set_xlabel('Số lần xuất hiện')
axes[1, 0].set_ylabel('Skill')

sns.histplot(df_salary['salary_avg'] / 1_000_000, bins=25, kde=True, ax=axes[1, 1], color='#b279a2')
axes[1, 1].set_title('Phân bố salary_avg trên các mẫu có lương')
axes[1, 1].set_xlabel('salary_avg (triệu VND/tháng)')
axes[1, 1].set_ylabel('Số tin')

salary_level_order = [lvl for lvl in ['Intern', 'Junior', 'Middle', 'Senior', 'Lead', 'Manager'] if lvl in df_salary['level'].dropna().unique()]
sns.boxplot(
    data=df_salary,
    x='level',
    y=df_salary['salary_avg'] / 1_000_000,
    order=salary_level_order,
    ax=axes[1, 2],
    color='#ff9da6'
)
axes[1, 2].set_title('Mức lương theo level (tập con có lương)')
axes[1, 2].set_xlabel('level')
axes[1, 2].set_ylabel('salary_avg (triệu VND)')
axes[1, 2].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

### Nhận xét nhanh từ thống kê đơn biến
- Có thể dùng `job_title_group` để quan sát trực tiếp nhóm nghề nào đang được tuyển nhiều.
- `location` cho phép nhận diện các trung tâm tuyển dụng nổi bật như Hà Nội, TP.HCM, Đà Nẵng.
- `level` phản ánh cấu trúc seniority của nhu cầu tuyển dụng.
- `skills_extracted` và các top skill là lõi của bài toán phân tích xu hướng công nghệ.
- Trên tập con có lương, phân bố `salary_avg` vẫn hữu ích để minh họa bối cảnh thị trường và so sánh sơ bộ theo `level`.

## 6. Làm sạch và chuẩn hóa dữ liệu
Để phân tích xu hướng chính xác, dữ liệu cần được làm sạch trước khi thống kê. Với đề tài này, cleaning tập trung vào các khía cạnh sau:
- Chuẩn hóa text để giảm nhiễu biểu diễn.
- Chuẩn hóa `location` và `remote_option` để gộp các nhãn cùng nghĩa.
- Chuẩn hóa `experience_years` để hỗ trợ suy ra `level` khi cần.
- Tạo `job_title_group`, `company_type` và loại bỏ duplicate bằng `job_id`.

Dù không dùng lương làm trung tâm, cleaning vẫn là bước quyết định để các biểu đồ xu hướng phản ánh đúng thực trạng dữ liệu.


In [ ]:
location_before = df_raw_merged['location'].fillna('Thiếu')
location_after = df_raw_merged['location'].apply(normalize_location).fillna('Thiếu')
remote_before = df_raw_merged['remote_option'].fillna('Thiếu')
remote_after = df_raw_merged['remote_option'].apply(normalize_remote_option).fillna('Thiếu')
experience_after = df_raw_merged['experience_years'].apply(parse_experience_years)

cleaning_summary = pd.DataFrame(
    {
        'Chỉ số': [
            'Số mẫu raw sau khi gộp',
            'Số mẫu sau cleaning',
            'Số mẫu bị loại do trùng lặp hoặc chuẩn hóa',
            'Số nhãn location khác nhau trước cleaning',
            'Số nhãn location khác nhau sau cleaning',
            'Số nhãn remote_option khác nhau trước cleaning',
            'Số nhãn remote_option khác nhau sau cleaning',
            'Số nhóm vị trí sau chuẩn hóa',
        ],
        'Giá trị': [
            len(df_raw_merged),
            len(df_clean),
            len(df_raw_merged) - len(df_clean),
            location_before.nunique(dropna=True),
            location_after.nunique(dropna=True),
            remote_before.nunique(dropna=True),
            remote_after.nunique(dropna=True),
            df_clean['job_title_group'].nunique(dropna=True),
        ],
    }
)
cleaning_summary


In [ ]:
location_compare = pd.concat(
    [
        location_before.value_counts().head(8).rename_axis('category').reset_index(name='count').assign(stage='Trước chuẩn hóa'),
        location_after.value_counts().head(8).rename_axis('category').reset_index(name='count').assign(stage='Sau chuẩn hóa'),
    ],
    ignore_index=True,
)

remote_compare = pd.concat(
    [
        remote_before.value_counts().rename_axis('category').reset_index(name='count').assign(stage='Trước chuẩn hóa'),
        remote_after.value_counts().rename_axis('category').reset_index(name='count').assign(stage='Sau chuẩn hóa'),
    ],
    ignore_index=True,
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(data=location_compare, x='count', y='category', hue='stage', ax=axes[0])
axes[0].set_title('So sánh phân bố location trước/sau cleaning')
axes[0].set_xlabel('Số tin')
axes[0].set_ylabel('location')

sns.barplot(data=remote_compare, x='category', y='count', hue='stage', ax=axes[1])
axes[1].set_title('So sánh phân bố remote_option trước/sau cleaning')
axes[1].set_xlabel('remote_option')
axes[1].set_ylabel('Số tin')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

experience_demo = pd.DataFrame(
    {
        'experience_raw': df_raw_merged['experience_years'].head(10),
        'experience_parsed': experience_after.head(10),
    }
)
experience_demo


### Ý nghĩa của bước cleaning
- Cleaning làm giảm số nhãn rời rạc ở `location` và `remote_option`, từ đó tránh chia nhỏ dữ liệu một cách giả tạo.
- `job_title_group` giúp gom nhiều job title chi tiết thành các nhóm nghề có thể so sánh được.
- `company_type` hỗ trợ quan sát xu hướng tuyển dụng theo loại hình doanh nghiệp.
- Loại trùng lặp là cần thiết để không làm sai lệch thống kê tần suất tuyển dụng và tần suất skill.


## 7. Mã hóa dữ liệu và xử lý ngôn ngữ tự nhiên
Dữ liệu tuyển dụng có rất nhiều trường text và category, nên cần đưa về dạng có thể phân tích được.

Trong project hiện tại, file `src/processing/extract_skills.py` đã thực hiện:
- tổng hợp nội dung từ `tech_stack`, `job_title` và các text liên quan;
- chuẩn hóa token, alias và loại bỏ nhiễu;
- trích xuất danh sách skill vào `skills_extracted`;
- tạo `skills_count` và các cột nhị phân như `has_python`, `has_java`, `has_js_ts`, `has_sql`, `has_cloud`, `has_data`, `has_devops`, `has_mobile`, `has_testing`, `has_ai`.

Nhờ vậy, dữ liệu text đã được biến đổi thành các đặc trưng có thể thống kê, trực quan hóa và dùng cho phân cụm nếu cần.


In [ ]:
feature_preview = df_features[
    [
        'job_title', 'job_title_group', 'location', 'level', 'company_type',
        'skills_extracted', 'skills_count', 'has_python', 'has_js_ts',
        'has_sql', 'has_cloud', 'has_data', 'has_devops'
    ]
].head(8)

display(feature_preview)

indicator_cols = [
    'has_ai', 'has_python', 'has_java', 'has_js_ts', 'has_sql',
    'has_cloud', 'has_data', 'has_devops', 'has_mobile', 'has_testing'
]
indicator_rate = df_features[indicator_cols].mean().sort_values(ascending=False).to_frame('Tỷ lệ xuất hiện')
indicator_rate


## 8. Xây dựng và lựa chọn đặc trưng
Với bài toán phân tích xu hướng tuyển dụng, các đặc trưng hữu ích nhất không phải là salary mà là các đặc trưng phản ánh **vai trò công việc, seniority, vị trí địa lý và kỹ năng công nghệ**.

Có thể chia thành 4 nhóm:
- **Nhóm nghề nghiệp**: `job_title_group`, `level`, `experience_years`.
- **Nhóm địa lý và ngữ cảnh doanh nghiệp**: `location`, `company_type`, `remote_option`, `source`.
- **Nhóm kỹ năng text/NLP**: `skills_extracted`, `skills_count`.
- **Nhóm chỉ báo skill**: `has_python`, `has_js_ts`, `has_sql`, `has_cloud`, `has_data`, `has_devops`, `has_ai`, ...

Đây là tập feature đủ mạnh để trả lời các câu hỏi xu hướng tuyển dụng, và cũng là nền phù hợp nếu muốn mở rộng sang clustering ở giai đoạn sau.


In [ ]:
feature_catalog = pd.DataFrame(
    [
        ['job_title_group', 'category', 'Nhóm vai trò công việc', 'Rất quan trọng'],
        ['level', 'ordinal/category', 'Cấp bậc tuyển dụng', 'Rất quan trọng'],
        ['experience_years', 'numeric', 'Số năm kinh nghiệm', 'Hữu ích'],
        ['location', 'category', 'Thành phố/khu vực tuyển dụng', 'Rất quan trọng'],
        ['company_type', 'category', 'Loại hình doanh nghiệp', 'Hữu ích'],
        ['remote_option', 'category', 'Hình thức làm việc', 'Hữu ích'],
        ['source', 'category', 'Nguồn dữ liệu tuyển dụng', 'Hữu ích để đối chiếu'],
        ['skills_extracted', 'text/list', 'Danh sách skill được trích xuất', 'Rất quan trọng'],
        ['skills_count', 'numeric', 'Số lượng skill trong tin tuyển dụng', 'Hữu ích'],
        ['has_python, has_js_ts, has_sql, ...', 'binary', 'Chỉ báo nhóm kỹ năng', 'Rất quan trọng'],
    ],
    columns=['Đặc trưng', 'Kiểu dữ liệu', 'Ý nghĩa', 'Đánh giá'],
)

role_skill_matrix = pd.crosstab(df_features['job_title_group'], df_features['has_python']).sort_values(1, ascending=False).head(10)

display(feature_catalog)
display(role_skill_matrix)


### Tập đặc trưng hữu ích đề xuất
Để trả lời tốt các câu hỏi nghiên cứu hiện tại, các đặc trưng nên được ưu tiên là:
- `job_title_group`
- `level`
- `experience_years`
- `location`
- `company_type`
- `remote_option`
- `source`
- `skills_extracted`
- `skills_count`
- các biến nhị phân kỹ năng như `has_python`, `has_js_ts`, `has_sql`, `has_cloud`, `has_data`, `has_devops`, `has_ai`

Đây là tập đặc trưng vừa giàu ý nghĩa nghiệp vụ, vừa đủ tốt để tiếp tục phân tích sâu hơn hoặc dùng cho clustering trong tương lai.


## 9. Trực quan hóa mối quan hệ đa biến
Phần này trả lời trực tiếp các câu hỏi trọng tâm của đề tài: nhóm nghề nào tuyển nhiều, kỹ năng nào đi cùng nhau, thành phố nào mạnh ở nhóm nghề nào, seniority level gắn với kỹ năng gì, và trên tập con có lương thì mức lương thay đổi ra sao theo level hoặc nhóm nghề.

In [ ]:
role_location = pd.crosstab(df_clean['job_title_group'], df_clean['location'])
role_location_top = role_location.loc[role_location.sum(axis=1).sort_values(ascending=False).head(10).index]
role_location_top = role_location_top[role_location_top.sum(axis=0).sort_values(ascending=False).head(8).index]

level_skill = df_features.groupby('level')[indicator_cols].mean().loc[
    [lvl for lvl in ['Intern', 'Junior', 'Middle', 'Senior', 'Lead', 'Manager'] if lvl in df_features['level'].dropna().unique()]
]

df_salary = df_clean.dropna(subset=['salary_avg']).copy()
top_salary_roles = df_salary['job_title_group'].value_counts().head(8).index

fig, axes = plt.subplots(2, 2, figsize=(17, 12))

sns.heatmap(role_location_top, cmap='YlGnBu', annot=True, fmt='g', ax=axes[0, 0])
axes[0, 0].set_title('Heatmap số tin theo job_title_group và location')
axes[0, 0].set_xlabel('location')
axes[0, 0].set_ylabel('job_title_group')

sns.heatmap(level_skill, cmap='OrRd', annot=True, fmt='.2f', ax=axes[0, 1])
axes[0, 1].set_title('Tỷ lệ xuất hiện skill indicator theo level')
axes[0, 1].set_xlabel('Skill indicator')
axes[0, 1].set_ylabel('level')

sns.countplot(
    data=df_clean,
    y='job_title_group',
    hue='source',
    order=df_clean['job_title_group'].value_counts().head(10).index,
    ax=axes[1, 0]
)
axes[1, 0].set_title('Top nhóm nghề theo từng nguồn')
axes[1, 0].set_xlabel('Số tin')
axes[1, 0].set_ylabel('job_title_group')

sns.boxplot(
    data=df_salary[df_salary['job_title_group'].isin(top_salary_roles)],
    x='job_title_group',
    y=df_salary[df_salary['job_title_group'].isin(top_salary_roles)]['salary_avg'] / 1_000_000,
    ax=axes[1, 1],
    color='#72b7b2'
)
axes[1, 1].set_title('Mức lương theo nhóm nghề (tập con có lương)')
axes[1, 1].set_xlabel('job_title_group')
axes[1, 1].set_ylabel('salary_avg (triệu VND)')
axes[1, 1].tick_params(axis='x', rotation=35)

plt.tight_layout()
plt.show()

In [ ]:
skill_lists = (
    df_features['skills_extracted']
    .fillna('')
    .str.split(', ')
    .apply(lambda items: [item for item in items if item])
)

pair_counter = Counter()
for items in skill_lists:
    unique_items = sorted(set(items))
    for pair in combinations(unique_items, 2):
        pair_counter[pair] += 1

top_pairs = pd.DataFrame(
    [
        {'skill_1': a, 'skill_2': b, 'count': c}
        for (a, b), c in pair_counter.most_common(15)
    ]
)
top_pairs


In [ ]:
if not top_pairs.empty:
    top_pairs_plot = top_pairs.copy()
    top_pairs_plot['pair'] = top_pairs_plot['skill_1'] + ' + ' + top_pairs_plot['skill_2']

    plt.figure(figsize=(12, 7))
    sns.barplot(data=top_pairs_plot, x='count', y='pair', color='#e45756')
    plt.title('Top cặp skill thường xuất hiện cùng nhau')
    plt.xlabel('Số lần đồng xuất hiện')
    plt.ylabel('Cặp skill')
    plt.tight_layout()
    plt.show()


### Diễn giải từ trực quan hóa đa biến
- Heatmap `job_title_group` theo `location` cho thấy thành phố nào mạnh ở nhóm nghề nào.
- Ma trận skill theo `level` giúp trả lời câu hỏi seniority nào gắn với các nhóm công nghệ nào.
- Top cặp skill đồng xuất hiện là bằng chứng trực tiếp cho các “combo công nghệ” phổ biến trên thị trường.
- Trên tập con có lương, boxplot theo `level` hoặc `job_title_group` giúp bổ sung góc nhìn về phân khúc nhân lực trên thị trường tuyển dụng IT.

## 10. Trực quan hóa không gian dữ liệu nhiều chiều
Mặc dù bài toán hiện tại là phân tích xu hướng, ta vẫn có thể giảm chiều dữ liệu đã mã hóa để nhìn cấu trúc tổng thể của không gian tuyển dụng IT. Điều này hữu ích nếu sau này muốn phát triển sang bài toán phân cụm.


In [ ]:
embedding_features = [
    'location', 'company_type', 'level', 'remote_option', 'job_title_group',
    'experience_years', 'skills_count', 'has_ai', 'has_python', 'has_java',
    'has_js_ts', 'has_sql', 'has_cloud', 'has_data', 'has_devops',
    'has_mobile', 'has_testing'
]

tsne_sample = df_features[embedding_features].copy()
if len(tsne_sample) > 400:
    tsne_sample = tsne_sample.sample(400, random_state=42)

categorical_cols = ['location', 'company_type', 'level', 'remote_option', 'job_title_group']
numeric_cols = [col for col in embedding_features if col not in categorical_cols]

preprocessor = ColumnTransformer(
    transformers=[
        (
            'cat',
            Pipeline(
                steps=[
                    ('imputer', SimpleImputer(strategy='most_frequent')),
                    ('onehot', OneHotEncoder(handle_unknown='ignore')),
                ]
            ),
            categorical_cols,
        ),
        (
            'num',
            Pipeline(
                steps=[
                    ('imputer', SimpleImputer(strategy='median')),
                    ('scaler', StandardScaler()),
                ]
            ),
            numeric_cols,
        ),
    ]
)

X_embedded = preprocessor.fit_transform(tsne_sample[embedding_features])
if hasattr(X_embedded, 'toarray'):
    X_embedded = X_embedded.toarray()

perplexity = min(30, max(5, len(tsne_sample) // 15))
coords = TSNE(
    n_components=2,
    perplexity=perplexity,
    learning_rate='auto',
    init='pca',
    random_state=42,
).fit_transform(X_embedded)

tsne_plot_df = tsne_sample[['level', 'job_title_group']].copy()
tsne_plot_df['tsne_1'] = coords[:, 0]
tsne_plot_df['tsne_2'] = coords[:, 1]

plt.figure(figsize=(12, 8))
sns.scatterplot(
    data=tsne_plot_df,
    x='tsne_1',
    y='tsne_2',
    hue='job_title_group',
    style='level',
    alpha=0.8,
    s=75,
)
plt.title('t-SNE trên không gian đặc trưng tuyển dụng IT')
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.tight_layout()
plt.show()


### Nhận xét về không gian nhiều chiều
- Nếu các điểm có xu hướng tụ lại theo `job_title_group` hoặc `level`, điều đó cho thấy dữ liệu có cấu trúc nghề nghiệp tương đối rõ.
- Đây là tín hiệu tích cực nếu sau này nhóm muốn đi tiếp sang bài toán phân cụm.
- Trong notebook hiện tại, t-SNE chủ yếu đóng vai trò minh họa cho cấu trúc nhiều chiều của dữ liệu đã mã hóa.


## 11. Đánh giá tính khả thi của bài toán
Dựa trên toàn bộ quy trình từ thu thập, cleaning, encoding đến trực quan hóa, có thể đánh giá mức độ khả thi của hướng phân tích xu hướng tuyển dụng như sau.


In [ ]:
feasibility_table = pd.DataFrame(
    [
        ['Tự thu thập dữ liệu', 'Đạt', 'Có crawler thực tế cho ITViec và TopCV'],
        ['Số lượng mẫu raw > 1000', 'Đạt', f'Tổng raw hiện có: {len(df_raw_merged)} mẫu'],
        ['Số lượng biến >= 5', 'Đạt', f'Raw data có {df_raw_merged.shape[1]} biến cốt lõi'],
        ['Có đặc trưng phù hợp để phân tích xu hướng', 'Đạt', 'Có job group, level, location, company type, remote option, skill'],
        ['Có dữ liệu text để làm NLP', 'Đạt', 'Có tech_stack và job_title để trích xuất kỹ năng'],
        ['Dữ liệu đủ sạch để EDA', 'Đạt', f'Clean data còn {len(df_clean)} mẫu sau loại trùng'],
        ['Có thể mở rộng sang clustering', 'Khả thi', 'Dữ liệu đã được mã hóa thành feature dạng category/binary/numeric'],
        ['Rủi ro dữ liệu', 'Cần lưu ý', 'Lệch nguồn, text ngắn ở một số tin, skill extraction vẫn có thể bỏ sót alias'],
    ],
    columns=['Tiêu chí', 'Đánh giá', 'Giải thích'],
)
feasibility_table


### Kết luận về tính khả thi
**Hướng phân tích xu hướng tuyển dụng là khả thi và phù hợp hơn so với việc lấy mức lương làm mục tiêu chính ở thời điểm hiện tại.**

Lý do chính:
- Dữ liệu raw vượt ngưỡng tối thiểu 1000 mẫu và được nhóm tự crawl.
- Các biến như `job_title_group`, `level`, `location`, `company_type`, `remote_option`, `skills_extracted` có ý nghĩa trực tiếp với bài toán xu hướng tuyển dụng.
- Dữ liệu text đã được chuẩn hóa và chuyển thành feature có thể phân tích bằng NLP mức cơ bản.
- Việc mức lương bị thiếu nhiều không còn là rào cản lớn nếu mục tiêu chính là phân tích xu hướng nghề nghiệp và công nghệ.
- Dù vậy, các mẫu có lương vẫn đủ để thực hiện **thống kê mô tả bổ sung về mức lương**, giúp bài làm đầy đủ và sát thực tế hơn.

Vì vậy, về mặt dữ liệu, hướng đi này vừa thực tế hơn, vừa bám sát chất lượng dữ liệu hiện có của nhóm.

## 12. Kết luận
Notebook đã chuyển trọng tâm từ bài toán dự đoán lương sang **phân tích xu hướng tuyển dụng IT tại Việt Nam**. Đây là hướng phù hợp hơn với dữ liệu crawl hiện tại, đặc biệt khi thông tin lương còn thiếu nhiều và không đủ ổn định để làm target chính.

Kết luận cuối cùng là:
- Bộ dữ liệu hiện tại phù hợp để khảo sát xu hướng tuyển dụng theo **nhóm nghề, địa điểm, seniority level, loại hình công ty và kỹ năng**.
- Các câu hỏi như “nhóm vị trí nào tuyển nhiều nhất”, “skill nào phổ biến nhất”, “skill nào thường đi cùng nhau”, “thành phố nào mạnh ở nhóm nghề nào” đều có thể trả lời bằng dữ liệu hiện có.
- **Thống kê mức lương vẫn nên được giữ lại** như một phần phân tích mô tả bổ sung trên tập con có công khai lương, thay vì dùng làm biến mục tiêu trung tâm.
- Tập đặc trưng được xây dựng trong `jobs_features.csv` là nền tốt cho cả EDA hiện tại lẫn các bước mở rộng như clustering trong tương lai.

## 13. Tài liệu tham khảo
- `docs/team_roles.md`
- `docs/problem_statement.md`
- `src/data_collection/itviec_crawler.py`
- `src/data_collection/topcv_crawler.py`
- `src/processing/clean_jobs.py`
- `src/processing/extract_skills.py`
- `data/raw/itviec_jobs_20260315_115039.csv`
- `data/raw/topcv_jobs_20260314_231436.csv`
- `data/clean-data/jobs_cleaned.csv`
- `data/clean-data/jobs_features.csv`
- Tài liệu tham khảo về feature engineering: <https://phamdinhkhanh.github.io/2019/01/07/Ky_thuat_feature_engineering.html>
- Tài liệu tham khảo về t-SNE: <https://www.datacamp.com/tutorial/introduction-t-sne>
